In [0]:
%sql
MERGE INTO catalog_ventas.gold.dim_fecha AS tgt
USING (
    SELECT DISTINCT
        CAST(DATE_FORMAT(vtafecha, 'yyyyMMdd') AS BIGINT) AS id_fecha,
        vtafecha,
        YEAR(vtafecha) AS anio,
        MONTH(vtafecha) AS mes,
        QUARTER(vtafecha) AS trimestre,
        CASE DAYOFWEEK(vtafecha)
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Lunes'
            WHEN 3 THEN 'Martes'
            WHEN 4 THEN 'Miercoles'
            WHEN 5 THEN 'Jueves'
            WHEN 6 THEN 'Viernes'
            WHEN 7 THEN 'Sabado'
        END AS dia_semana,
        CASE 
            WHEN DAYOFWEEK(vtafecha) IN (1,7) THEN TRUE 
            ELSE FALSE 
        END AS es_fin_de_semana
    FROM catalog_ventas.silver.ventas_clean
    WHERE vtafecha IS NOT NULL
) AS src
ON tgt.id_fecha = src.id_fecha

WHEN MATCHED THEN UPDATE SET
    tgt.vtafecha = src.vtafecha,
    tgt.anio = src.anio,
    tgt.mes = src.mes,
    tgt.trimestre = src.trimestre,
    tgt.dia_semana = src.dia_semana,
    tgt.es_fin_de_semana = src.es_fin_de_semana

WHEN NOT MATCHED THEN INSERT (
    id_fecha,
    vtafecha,
    anio,
    mes,
    trimestre,
    dia_semana,
    es_fin_de_semana
)
VALUES (
    src.id_fecha,
    src.vtafecha,
    src.anio,
    src.mes,
    src.trimestre,
    src.dia_semana,
    src.es_fin_de_semana
);